# Chuẩn bị dữ liệu demo HairCare Insight

Notebook tách riêng hai mục tiêu: catalog cho môn Web lấy tên/giá USD từ `products_crawl.csv`; dữ liệu cho môn ML lấy Quantity/Price/doanh thu từ Online Retail II. Với ML, chỉ đổi nhãn SKU/danh mục sang haircare giả lập; giữ nguyên số lượng và giá giao dịch nguồn. Doanh thu vẫn là `Quantity × Price` theo tiền tệ nguồn GBP, không dùng giá crawl USD.

## Nguyên tắc và giới hạn

- Hai file nguồn không bị thay đổi. Toàn bộ đầu ra nằm trong `processed/`.
- Dataset giao dịch là bán lẻ tổng hợp, không phải haircare. Mapping StockCode → sản phẩm haircare chỉ đổi nhãn để trình diễn; **không được diễn giải thành doanh số haircare thật**.
- Taxonomy chuẩn dùng 6 nhóm: `shampoo`, `hairmask`, `treatment`, `dry shampoo`, `conditioner`, `hairstyling`. `Treatment / Oil` và sản phẩm `Styling & Finishing` có chữ `serum` trong tên đều map vào `treatment`; các nhãn nguồn còn lại được giữ theo nhóm tương ứng.
- Tên, mã và giá trong catalog lấy từ `products_crawl.csv`; cột giá sản phẩm dùng USD theo trường `currency`. Tồn đầu kỳ, lead time và safety stock là giả định mô phỏng.
- Web demo: giả lập cả 59 sản phẩm đều available; trạng thái crawl gốc được lưu riêng ở `source_availability` để truy vết.
- ML: giữ nguyên giá và quantity của dataset giao dịch (nguồn Online Retail II, GBP); gán lần lượt các SKU nguồn duy nhất vào 59 sản phẩm crawl để có nhiều chuỗi bán cho demo tồn kho. Không ghép giá USD từ crawl vào doanh thu ML.
- Mapping tên/danh mục chỉ là nhãn giả lập; dữ liệu model thể hiện mẫu bán lẻ nguồn, không phải doanh số haircare thực tế. Không dùng kết quả để khẳng định nhu cầu haircare ngoài phạm vi mô phỏng.
- Web: giá `products_crawl.csv` là USD (file có `currency=USD` ở mọi dòng); catalog Web được xuất riêng và không tham gia tính doanh thu ML.
- Quantity ngoại lệ được giữ nguyên trong dữ liệu ML; chỉ gắn cờ để nhóm đánh giá. Không cap biến mục tiêu trước khi huấn luyện hoặc tính doanh thu.

Chạy lần lượt từ trên xuống. Cell xuất file có tính lặp lại và sẽ ghi đè các file cùng tên trong thư mục đầu ra.

In [ ]:
from pathlib import Path
import hashlib
import re
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        for data_root in [candidate, candidate / 'data' / 'ml']:
            source_root = data_root / 'raw' if (data_root / 'raw' / 'products_crawl.csv').is_file() else data_root
            if (source_root / 'ecomcosmetic').is_dir() and (source_root / 'online+retail+ii (1)').is_dir() and (source_root / 'products_crawl.csv').is_file():
                return data_root
    raise FileNotFoundError('Không tìm thấy hai thư mục dataset. Hãy đặt notebook trong project hoặc thư mục con của project.')

ROOT = find_project_root()
SOURCE_DIR = ROOT / 'raw' if (ROOT / 'raw' / 'products_crawl.csv').is_file() else ROOT
COSMETIC_PATH = SOURCE_DIR / 'ecomcosmetic' / 'E-commerce  cosmetic dataset.csv'
PRODUCTS_CRAWL_PATH = SOURCE_DIR / 'products_crawl.csv'
RETAIL_PATH = SOURCE_DIR / 'online+retail+ii (1)' / 'online_retail_II.xlsx'
OUTPUT_DIR = ROOT / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Threshold chỉ dùng gắn cờ ngoại lệ; không cắt hay thay đổi Quantity.
OUTLIER_QUANTITY_THRESHOLD = 100
CATEGORY_ORDER = ['shampoo', 'hairmask', 'treatment', 'dry shampoo', 'conditioner', 'hairstyling']
CATEGORY_VI = {
    'shampoo': 'Dầu gội', 'hairmask': 'Ủ tóc', 'treatment': 'Dưỡng tóc',
    'dry shampoo': 'Dầu gội khô',
    'conditioner': 'Dầu xả', 'hairstyling': 'Tạo kiểu tóc',
}

print('Project:', ROOT)
print('Output:', OUTPUT_DIR)
print('Nguồn crawl:', PRODUCTS_CRAWL_PATH.exists(), '| CSV tham khảo:', COSMETIC_PATH.exists(), '| XLSX:', RETAIL_PATH.exists())

## 1. Chuẩn hóa catalog crawl, dùng USD và đối chiếu taxonomy

Catalog chính của website lấy từ `products_crawl.csv`: giữ product ID, tên, thương hiệu, size, URL/ảnh và giá `price_usd` sau khi xác nhận toàn bộ `currency=USD`. Taxonomy crawl được ánh xạ vào 6 nhóm chuẩn; `Treatment / Oil` và sản phẩm serum được gộp vào `treatment`. File cosmetic ban đầu vẫn làm nguồn tham khảo riêng, không trộn giá vì đơn vị của nó không chắc chắn.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = unicodedata.normalize('NFKC', str(value)).replace('\ufffd', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else pd.NA

cosmetics = pd.read_csv(COSMETIC_PATH, encoding='cp1252', low_memory=False)
required_columns = {'product_name', 'website', 'country', 'category', 'subcategory', 'price', 'brand'}
missing_columns = required_columns - set(cosmetics.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột trong CSV: {sorted(missing_columns)}')

catalog = cosmetics.copy()
catalog.insert(0, 'source_row_id', np.arange(2, len(catalog) + 2))
text_columns = ['product_name', 'website', 'country', 'category', 'subcategory', 'brand', 'ingredients', 'form', 'type', 'size']
for column in text_columns:
    if column in catalog.columns:
        catalog[f'{column}_source'] = catalog[column]
        catalog[column] = catalog[column].map(clean_text).astype('string')

catalog['encoding_warning'] = catalog['product_name_source'].astype('string').str.contains('\ufffd', regex=False, na=False)
catalog['price'] = pd.to_numeric(catalog['price'], errors='coerce')
catalog['rating'] = pd.to_numeric(catalog.get('rating'), errors='coerce')
catalog['noofratings'] = pd.to_numeric(
    catalog.get('noofratings', pd.Series(index=catalog.index, dtype='string')).astype('string').str.replace(',', '', regex=False),
    errors='coerce',
)
catalog['size_numeric'] = pd.to_numeric(catalog['size'].astype('string').str.extract(r'(\d+(?:\.\d+)?)', expand=False), errors='coerce')

hair = catalog.loc[catalog['category'].str.casefold().eq('hair').fillna(False)].copy()
hair = hair.loc[hair['product_name'].notna() & hair['brand'].notna()].copy()
hair['name_norm'] = hair['product_name'].str.casefold().str.replace(r'[^\w]+', ' ', regex=True).str.strip()
hair['brand_norm'] = hair['brand'].str.casefold().str.replace(r'[^\w]+', ' ', regex=True).str.strip()
hair['size_norm'] = hair['size'].fillna('').str.casefold().str.replace(r'\s+', '', regex=True)

category_labels = {
    'shampoo': 'shampoo',
    'hairmask': 'hairmask',
    'oil': 'treatment',
    'serum': 'treatment',
    'dry shampoo': 'dry shampoo',
    'conditioner': 'conditioner',
    'hairstyling': 'hairstyling',
}
hair['subcategory_normalized'] = hair['subcategory'].str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip()
hair['suggested_haircare_category'] = hair['subcategory_normalized'].map(category_labels).astype('string')
hair['category_mapping_method'] = np.where(
    hair['subcategory_normalized'].isin(['oil', 'serum']),
    'Oil/serum subcategory merged into treatment',
    np.where(hair['suggested_haircare_category'].notna(), 'Direct mapping from dataset subcategory', 'Cần rà soát thủ công'),
)
hair['needs_manual_review'] = (
    hair['suggested_haircare_category'].isna() | hair['price'].isna() | hair['price'].le(0) | hair['encoding_warning']
)
hair['source_price_unit'] = 'Chưa xác minh; giữ nguyên giá nguồn'

hair['completeness_score'] = (
    hair['price'].notna().astype(int) + hair['ingredients'].notna().astype(int) +
    hair['rating'].notna().astype(int) + (~hair['encoding_warning']).astype(int)
)
hair = hair.sort_values(
    ['completeness_score', 'noofratings', 'source_row_id'],
    ascending=[False, False, True], kind='stable', na_position='last',
)
hair['duplicate_listing'] = hair.duplicated(['name_norm', 'brand_norm', 'size_norm'], keep='first')
hair_clean = hair.loc[~hair['duplicate_listing']].copy()
hair_clean['catalog_id'] = hair_clean.apply(
    lambda row: 'HC-CAT-' + hashlib.sha1(
        f"{row['name_norm']}|{row['brand_norm']}|{row['size_norm']}".encode('utf-8')
    ).hexdigest()[:12].upper(), axis=1,
)

catalog_summary = pd.DataFrame([
    ('CSV ban đầu', len(cosmetics)),
    ('Listing category hair trước khử trùng', len(hair)),
    ('Listing trùng đã loại', int(hair['duplicate_listing'].sum())),
    ('Listing hair sau làm sạch', len(hair_clean)),
    ('Cần rà soát thủ công', int(hair_clean['needs_manual_review'].sum())),
    ('Giá thiếu/không hợp lệ', int((hair_clean['price'].isna() | hair_clean['price'].le(0)).sum())),
    ('Tên có dấu hiệu lỗi mã hóa', int(hair_clean['encoding_warning'].sum())),
], columns=['Chỉ số', 'Số lượng'])
display(catalog_summary)
display(hair_clean['suggested_haircare_category'].fillna('Cần rà soát').value_counts().rename_axis('Danh mục nguồn chuẩn hóa').to_frame('Listing'))
display(hair_clean[['catalog_id', 'product_name', 'brand', 'subcategory', 'suggested_haircare_category', 'price', 'country', 'website', 'needs_manual_review']].head(12))

# Catalog chính cho website được lấy từ dữ liệu crawl; giữ nguyên giá USD, không quy đổi.
products_crawl = pd.read_csv(PRODUCTS_CRAWL_PATH, encoding='utf-8-sig', low_memory=False)
crawl_required = {'product_id', 'product_name', 'brand', 'category', 'price', 'currency', 'availability'}
missing_crawl_columns = crawl_required - set(products_crawl.columns)
if missing_crawl_columns:
    raise ValueError(f'Thiếu cột trong products_crawl.csv: {sorted(missing_crawl_columns)}')

crawl_catalog = products_crawl.copy()
crawl_catalog['product_id'] = crawl_catalog['product_id'].astype('string').str.replace(r'\.0$', '', regex=True).str.strip()
for column in ['product_name', 'brand', 'category', 'size', 'product_description', 'key_benefits', 'ingredients', 'how_to_use', 'product_url', 'source_url', 'image_url', 'image_local_path']:
    if column in crawl_catalog.columns:
        crawl_catalog[column] = crawl_catalog[column].map(clean_text).astype('string')
crawl_catalog['currency'] = crawl_catalog['currency'].astype('string').str.strip().str.upper()
crawl_catalog['price_usd'] = pd.to_numeric(crawl_catalog['price'], errors='coerce')
crawl_catalog['source_availability'] = crawl_catalog['availability'].astype('boolean')
crawl_catalog['availability'] = True
crawl_catalog['availability_is_simulated'] = True
if crawl_catalog['product_id'].isna().any() or crawl_catalog['product_id'].duplicated().any():
    raise ValueError('product_id trong products_crawl.csv phải tồn tại và duy nhất.')
if set(crawl_catalog['currency'].dropna().unique()) != {'USD'}:
    raise ValueError('Currency trong products_crawl.csv không đồng nhất USD; cần kiểm tra trước khi dùng giá.')
if crawl_catalog['price_usd'].isna().any() or crawl_catalog['price_usd'].le(0).any():
    raise ValueError('Có giá USD bị thiếu hoặc không hợp lệ trong products_crawl.csv.')

crawl_category_map = {
    'Shampoo': 'shampoo',
    'Hair Masks': 'hairmask',
    'Treatment / Oil': 'treatment',
    'Dry Shampoo': 'dry shampoo',
    'Conditioner': 'conditioner',
    'Styling & Finishing': 'hairstyling',
}
crawl_catalog['category_source'] = crawl_catalog['category']
crawl_catalog['category'] = crawl_catalog['category_source'].map(crawl_category_map).astype('string')
serum_in_styling = (
    crawl_catalog['category_source'].eq('Styling & Finishing') &
    crawl_catalog['product_name'].str.contains(r'\bserum\b', case=False, regex=True, na=False)
)
crawl_catalog.loc[serum_in_styling, 'category'] = 'treatment'
if 'categories' in crawl_catalog.columns:
    crawl_catalog['categories'] = crawl_catalog['category']
crawl_catalog['category_vi'] = crawl_catalog['category'].map(CATEGORY_VI).astype('string')
crawl_catalog['category_mapping_rule'] = np.select(
    [serum_in_styling, crawl_catalog['category_source'].eq('Treatment / Oil')],
    ['Styling & Finishing + product name contains serum -> treatment', 'Treatment / Oil -> treatment'],
    default='Direct source category mapping',
)
if crawl_catalog['category'].isna().any() or set(crawl_catalog['category'].unique()) != set(CATEGORY_ORDER):
    raise ValueError('Taxonomy crawl chưa ánh xạ đủ đúng 6 danh mục HairCare.')
crawl_catalog['price_unit'] = 'USD'
crawl_catalog['catalog_source'] = 'products_crawl.csv'
crawl_catalog['needs_manual_review'] = False
crawl_catalog['catalog_id'] = 'HC-' + crawl_catalog['product_id']
crawl_catalog = crawl_catalog.sort_values(
    ['category', 'product_name', 'product_id'],
    key=lambda col: col.map({name: position for position, name in enumerate(CATEGORY_ORDER)}) if col.name == 'category' else col,
    kind='stable',
).reset_index(drop=True)

if set(crawl_catalog['currency']) != {'USD'} or set(crawl_catalog['price_unit']) != {'USD'}:
    raise ValueError('Kiểm tra đơn vị giá thất bại: catalog website phải được định giá bằng USD.')
display(crawl_catalog.groupby(['category', 'category_vi'], as_index=False).agg(
    products=('product_id', 'size'), available=('availability', 'sum'),
    min_price_usd=('price_usd', 'min'), max_price_usd=('price_usd', 'max'),
))
display(crawl_catalog[['product_id', 'product_name', 'brand', 'category', 'price_usd', 'currency', 'availability']].head(12))

## 2. Làm sạch giao dịch bán lẻ và kiểm tra chất lượng

Chuẩn hóa mã hóa đơn/SKU, ngày giờ, số lượng và giá; bỏ invoice trùng giữa hai sheet theo kết quả khảo sát trước; loại khỏi tập **giao dịch bán hợp lệ** các dòng hủy, hoàn/điều chỉnh có Quantity ≤ 0, Price ≤ 0 hoặc thiếu khóa/ngày. Dòng quantity cực lớn được đánh dấu và giữ nguyên số liệu nguồn.

In [ ]:
excel_file = pd.ExcelFile(RETAIL_PATH, engine='openpyxl')
expected_columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
seen_invoice_ids = set()
transaction_frames = []
retail_quality_rows = []

for sheet_name in excel_file.sheet_names:
    frame = pd.read_excel(excel_file, sheet_name=sheet_name, usecols=expected_columns, engine='openpyxl')
    source_row_count = len(frame)
    frame['Invoice'] = frame['Invoice'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True).str.upper()
    frame['StockCode'] = frame['StockCode'].astype('string').str.strip().str.upper()
    frame['Description'] = frame['Description'].map(clean_text).astype('string')
    frame['InvoiceDate'] = pd.to_datetime(frame['InvoiceDate'], errors='coerce')
    frame['Quantity'] = pd.to_numeric(frame['Quantity'], errors='coerce')
    frame['Price'] = pd.to_numeric(frame['Price'], errors='coerce')
    frame['Country'] = frame['Country'].map(clean_text).astype('string')

    duplicate_invoice_mask = frame['Invoice'].isin(seen_invoice_ids)
    duplicate_invoice_ids = int(frame.loc[duplicate_invoice_mask, 'Invoice'].nunique())
    duplicate_invoice_rows = int(duplicate_invoice_mask.sum())
    frame = frame.loc[~duplicate_invoice_mask].copy()
    seen_invoice_ids.update(frame['Invoice'].dropna().unique().tolist())

    cancelled = frame['Invoice'].str.startswith('C', na=False)
    invalid_quantity = frame['Quantity'].isna() | frame['Quantity'].le(0)
    invalid_price = frame['Price'].isna() | frame['Price'].le(0)
    invalid_date = frame['InvoiceDate'].isna()
    invalid_sku = frame['StockCode'].isna() | frame['StockCode'].eq('')
    valid_sale = ~(cancelled | invalid_quantity | invalid_price | invalid_date | invalid_sku)
    sales = frame.loc[valid_sale].copy()
    sales['revenue_source'] = sales['Quantity'] * sales['Price']
    sales['sale_date'] = sales['InvoiceDate'].dt.floor('D')
    sales['quantity_outlier_gt_100'] = sales['Quantity'].gt(OUTLIER_QUANTITY_THRESHOLD)
    transaction_frames.append(sales[['Invoice', 'StockCode', 'Description', 'Country', 'InvoiceDate', 'sale_date', 'Quantity', 'Price', 'revenue_source', 'quantity_outlier_gt_100']])

    retail_quality_rows.append({
        'sheet': sheet_name,
        'source_rows': source_row_count,
        'duplicate_invoice_ids_removed': duplicate_invoice_ids,
        'duplicate_invoice_rows_removed': duplicate_invoice_rows,
        'cancelled_rows_excluded': int(cancelled.sum()),
        'quantity_nonpositive_or_missing_excluded': int(invalid_quantity.sum()),
        'price_nonpositive_or_missing_excluded': int(invalid_price.sum()),
        'date_missing_excluded': int(invalid_date.sum()),
        'sku_missing_excluded': int(invalid_sku.sum()),
        'valid_sales_rows': len(sales),
        'missing_customer_id_in_source': int(frame['Customer ID'].isna().sum()),
    })
    del frame, sales

retail_quality = pd.DataFrame(retail_quality_rows)
transactions_clean = pd.concat(transaction_frames, ignore_index=True)
transactions_clean['revenue_source'] = transactions_clean['revenue_source'].round(2)
display(retail_quality)
print(f"Giao dịch hợp lệ sau làm sạch: {len(transactions_clean):,}")
print('Khoảng ngày:', transactions_clean['sale_date'].min().date(), '→', transactions_clean['sale_date'].max().date())
print('SKU có bán:', transactions_clean['StockCode'].nunique())
print(f'Quantity > {OUTLIER_QUANTITY_THRESHOLD} được giữ nguyên và gắn cờ:', int(transactions_clean['quantity_outlier_gt_100'].sum()))

## 3. Tạo mapping sản phẩm haircare giả lập và chuỗi bán demo

Cả 59 sản phẩm crawl được giả lập là available cho web demo và ghép một-một với một SKU nguồn đủ điều kiện, theo thứ tự sản phẩm cố định và số hóa đơn nguồn giảm dần. Trạng thái availability gốc được lưu ở `source_availability`; trạng thái `availability` trong catalog đầu ra là trạng thái mô phỏng. Các cột product ID/tên/category/giá USD là của sản phẩm crawl; StockCode và lịch sử bán vẫn đến từ dataset tổng quát. Vì thế doanh số ghép vào từng sản phẩm vẫn là **proxy mô phỏng**, không phải lịch sử bán thật của sản phẩm đó.

In [ ]:
non_product_pattern = r'POSTAGE|CARRIAGE|BANK CHARGES|DOTCOM|AMAZON FEE|ADJUST|MANUAL|SAMPLES|CRUK'
transactions_clean['description_is_service_or_adjustment'] = transactions_clean['Description'].str.upper().str.contains(non_product_pattern, regex=True, na=False)
product_candidates = transactions_clean.loc[
    transactions_clean['Description'].notna() & ~transactions_clean['description_is_service_or_adjustment']
].copy()

sku_activity = product_candidates.groupby('StockCode', as_index=False).agg(
    source_description=('Description', 'first'),
    distinct_invoices=('Invoice', 'nunique'),
    source_rows=('Invoice', 'size'),
    source_median_price=('Price', 'median'),
    source_total_quantity=('Quantity', 'sum'),
)
sku_activity = sku_activity.sort_values(
    ['distinct_invoices', 'source_rows', 'StockCode'],
    ascending=[False, False, True], kind='stable',
).reset_index(drop=True)
demo_products = crawl_catalog.copy()
source_category_order = {'Shampoo': 0, 'Hair Masks': 1, 'Treatment / Oil': 2, 'Dry Shampoo': 4, 'Conditioner': 5, 'Styling & Finishing': 6}
demo_products['_category_order'] = demo_products['category_source'].map(source_category_order)
mapping_serum_mask = demo_products['category_source'].eq('Styling & Finishing') & demo_products['product_name'].str.contains(r'\bserum\b', case=False, regex=True, na=False)
demo_products.loc[mapping_serum_mask, '_category_order'] = 3
demo_products = demo_products.sort_values(
    ['_category_order', 'product_name', 'product_id'], kind='stable',
).drop(columns='_category_order').reset_index(drop=True)
sku_activity = sku_activity.sort_values(
    ['distinct_invoices', 'source_rows', 'StockCode'],
    ascending=[False, False, True], kind='stable',
).head(len(demo_products)).reset_index(drop=True)
if len(sku_activity) != len(demo_products):
    raise ValueError(f'Không đủ SKU nguồn: cần {len(demo_products)}, có {len(sku_activity)}.')

mapping = pd.concat([
    demo_products[['product_id', 'product_name', 'category', 'category_vi']].reset_index(drop=True),
    sku_activity.reset_index(drop=True),
], axis=1)
mapping['source_median_price_gbp'] = mapping['source_median_price'].round(2)
mapping['source_currency'] = 'GBP'
mapping['source_dataset'] = 'Online Retail II'
mapping['mapping_type'] = 'SIMULATED LABEL ONLY - source quantities/prices unchanged; not actual HairCare sales'
mapping['mapping_rule'] = 'All crawled products paired one-to-one with top source SKUs by distinct invoice count; stable order; web availability is simulated'
mapping['price_used_for_ml'] = 'Original transaction Price in GBP; products_crawl price excluded'
if mapping['product_id'].duplicated().any() or mapping['StockCode'].duplicated().any():
    raise ValueError('Mapping ML phải là quan hệ một-một giữa sản phẩm HairCare và SKU nguồn.')
display(mapping[['product_id', 'product_name', 'category', 'StockCode', 'source_description', 'distinct_invoices', 'source_median_price_gbp', 'source_currency', 'mapping_type']].head(15))

mapped_transactions = product_candidates.merge(
    mapping[['StockCode', 'product_id', 'category', 'category_vi', 'product_name']],
    on='StockCode', how='inner', validate='many_to_one',
)
mapped_transactions = mapped_transactions.rename(columns={
    'Quantity': 'quantity_sold',
    'Price': 'unit_price_gbp',
    'revenue_source': 'revenue_gbp',
})
mapped_transactions['quantity_outlier_gt_100'] = mapped_transactions['quantity_sold'].gt(OUTLIER_QUANTITY_THRESHOLD)
mapped_transactions['currency'] = 'GBP'

daily_observed = mapped_transactions.groupby(['sale_date', 'product_id'], as_index=False).agg(
    quantity_sold=('quantity_sold', 'sum'),
    revenue_gbp=('revenue_gbp', 'sum'),
    source_order_count=('Invoice', 'nunique'),
    unit_price_gbp=('unit_price_gbp', 'mean'),
    outlier_source_lines=('quantity_outlier_gt_100', 'sum'),
)

all_dates = pd.date_range(transactions_clean['sale_date'].min(), transactions_clean['sale_date'].max(), freq='D', name='sale_date')
all_product_ids = mapping['product_id'].tolist()
complete_index = pd.MultiIndex.from_product([all_dates, all_product_ids], names=['sale_date', 'product_id'])
daily_demo = daily_observed.set_index(['sale_date', 'product_id']).reindex(complete_index).reset_index()
daily_demo = daily_demo.merge(
    mapping[['product_id', 'category', 'category_vi', 'product_name']],
    on='product_id', how='left', validate='many_to_one',
)
for column in ['quantity_sold', 'revenue_gbp', 'source_order_count', 'outlier_source_lines']:
    daily_demo[column] = daily_demo[column].fillna(0)
daily_demo['currency'] = 'GBP'
daily_demo['is_zero_sale_day'] = daily_demo['quantity_sold'].eq(0)
daily_demo['quantity_sold'] = daily_demo['quantity_sold'].astype('int64')
daily_demo['source_order_count'] = daily_demo['source_order_count'].astype('int64')
daily_demo['outlier_source_lines'] = daily_demo['outlier_source_lines'].astype('int64')
daily_demo['revenue_gbp'] = daily_demo['revenue_gbp'].round(2)

print(f"SKU nguồn được gán nhãn giả lập: {mapping['StockCode'].nunique()} / {len(demo_products)} sản phẩm crawl; tất cả available trong web demo")
print(f"Dòng product-ngày ML (ngày không bán điền 0): {len(daily_demo):,}")
print(f"Dòng giao dịch được map: {len(mapped_transactions):,}")
print(f"Dòng Quantity > {OUTLIER_QUANTITY_THRESHOLD} được giữ nguyên và gắn cờ: {int(mapped_transactions['quantity_outlier_gt_100'].sum()):,}")
display(daily_demo.groupby(['category', 'category_vi'], as_index=False).agg(
    products=('product_id', 'nunique'), days=('sale_date', 'nunique'), quantity_sold=('quantity_sold', 'sum'), revenue_gbp=('revenue_gbp', 'sum'),
    zero_sale_days=('is_zero_sale_day', 'sum'),
))

## 4. Kiểm tra nhanh, trực quan hóa và xuất dữ liệu đã chuẩn bị

`quantity_sold` và `revenue_gbp` là Quantity và Quantity × Price từ Online Retail II sau lọc giao dịch; không nhân giá USD crawl. Chỉ tên/mã/danh mục sản phẩm là nhãn HairCare. Catalog Web USD được xuất riêng, không tham gia tính doanh thu ML.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for product_id, group in daily_demo.groupby('product_id'):
    ax.plot(group['sale_date'], group['quantity_sold'].rolling(14, min_periods=1).mean(), label=f"{group['product_name'].iloc[0]} ({product_id})", linewidth=1.1)
ax.set_title('Quantity nguồn theo ngày - nhãn HairCare giả lập (trung bình trượt 14 ngày)')
ax.set_xlabel('Ngày')
ax.set_ylabel('Đơn vị nguồn/ngày')
ax.legend(ncol=2, fontsize=7)
ax.grid(alpha=0.25)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

quality_report = pd.DataFrame([
    {'dataset': 'products_crawl.csv', 'metric': 'products_cleaned', 'value': len(crawl_catalog), 'note': 'Catalog sản phẩm nhóm crawl; giá giữ nguyên USD'},
    {'dataset': 'products_crawl.csv', 'metric': 'categories_mapped', 'value': crawl_catalog['category'].nunique(), 'note': '6 nhóm taxonomy; Treatment / Oil và Styling & Finishing serum map vào treatment'},
    {'dataset': 'products_crawl.csv', 'metric': 'source_products_unavailable', 'value': int((~crawl_catalog['source_availability'].fillna(False)).sum()), 'note': 'Trạng thái availability theo dữ liệu crawl gốc'},
    {'dataset': 'Web demo', 'metric': 'products_available_in_demo', 'value': int(crawl_catalog['availability'].sum()), 'note': 'Giả lập tất cả sản phẩm đều available'},
    {'dataset': 'Web demo', 'metric': 'products_unavailable_in_demo', 'value': int((~crawl_catalog['availability']).sum()), 'note': 'Không có sản phẩm hết hàng trong demo'},
    {'dataset': 'Cosmetic CSV tham khảo', 'metric': 'hair_listings_cleaned', 'value': len(hair_clean), 'note': 'Nguồn tham khảo riêng, đơn vị giá chưa xác minh'},
    {'dataset': 'Online Retail II', 'metric': 'valid_sales_rows', 'value': len(transactions_clean), 'note': 'Sau loại invoice trùng, hủy, hoàn và dữ liệu khóa/giá/ngày lỗi'},
    {'dataset': 'Online Retail II', 'metric': 'duplicate_invoice_rows_removed', 'value': int(retail_quality['duplicate_invoice_rows_removed'].sum()), 'note': 'Invoice bị lặp giữa các sheet'},
    {'dataset': 'Online Retail II', 'metric': 'demo_products_mapped', 'value': len(mapping), 'note': 'Tất cả sản phẩm crawl; lịch sử giao dịch được gán giả lập'},
    {'dataset': 'Online Retail II', 'metric': 'quantity_outlier_lines_kept', 'value': int(mapped_transactions['quantity_outlier_gt_100'].sum()), 'note': 'Quantity ngoại lệ được giữ nguyên; chỉ gắn cờ kiểm tra'},
    {'dataset': 'Online Retail II', 'metric': 'daily_ml_rows', 'value': len(daily_demo), 'note': 'Panel đủ ngày × SKU được gán nhãn; ngày không giao dịch điền 0'},
])

catalog_output = crawl_catalog.copy()
hair_reference_output = hair_clean.drop(columns=['completeness_score', 'duplicate_listing'], errors='ignore')
catalog_output_path = OUTPUT_DIR / 'haircare_catalog_cleaned.csv'
hair_reference_output_path = OUTPUT_DIR / 'cosmetic_hair_reference_cleaned.csv'
transactions_output_path = OUTPUT_DIR / 'retail_transactions_cleaned.csv.gz'
mapping_output_path = OUTPUT_DIR / 'haircare_demo_product_mapping.csv'
daily_output_path = OUTPUT_DIR / 'haircare_demo_daily_sales.csv'
quality_output_path = OUTPUT_DIR / 'preparation_quality_report.csv'
retail_quality_output_path = OUTPUT_DIR / 'retail_cleaning_report.csv'

catalog_output.to_csv(catalog_output_path, index=False, encoding='utf-8-sig')
hair_reference_output.to_csv(hair_reference_output_path, index=False, encoding='utf-8-sig')
transactions_clean.to_csv(transactions_output_path, index=False, compression='gzip', encoding='utf-8-sig')
mapping.to_csv(mapping_output_path, index=False, encoding='utf-8-sig')
daily_demo.to_csv(daily_output_path, index=False, encoding='utf-8-sig')
quality_report.to_csv(quality_output_path, index=False, encoding='utf-8-sig')
retail_quality.to_csv(retail_quality_output_path, index=False, encoding='utf-8-sig')

display(quality_report)
print('\nĐã xuất:')
for path in [catalog_output_path, hair_reference_output_path, transactions_output_path, mapping_output_path, daily_output_path, quality_output_path, retail_quality_output_path]:
    print(f'- {path.relative_to(ROOT)} ({path.stat().st_size / 1024:,.1f} KB)')

## 5. Các file đầu ra và cách dùng ở bước tiếp theo

- `processed/haircare_catalog_cleaned.csv`: catalog 59 sản phẩm crawl; giả lập cả 59 sản phẩm available cho web demo. `source_availability` lưu trạng thái crawl gốc để truy vết; `availability` là trạng thái demo.
- `processed/cosmetic_hair_reference_cleaned.csv`: dữ liệu hair từ CSV cosmetic ban đầu, giữ làm nguồn tham khảo riêng; không trộn giá vì đơn vị chưa xác minh.
- `processed/retail_transactions_cleaned.csv.gz`: toàn bộ giao dịch hợp lệ sau làm sạch, nén gzip; giữ StockCode/Quantity/Price nguồn và không chứa Customer ID. Đọc bằng `pd.read_csv(path)` (pandas tự nhận biết gzip).
- `processed/haircare_demo_product_mapping.csv`: mapping 59 StockCode nguồn sang product ID/tên/danh mục crawl. Giá crawl USD không tham gia mapping giá ML; giữ StockCode và mô tả gốc để truy vết.
- `processed/haircare_demo_daily_sales.csv`: panel ngày × 59 SKU đã gán nhãn, ngày không bán điền 0; `quantity_sold` và `revenue_gbp` là mục tiêu từ giao dịch nguồn, có thể dùng cho bước ML demo.
- `processed/preparation_quality_report.csv` và `retail_cleaning_report.csv`: số liệu kiểm toán các bước làm sạch.

Trước khi huấn luyện: chia train/test theo thời gian; tạo lag/rolling chỉ từ quá khứ; so sánh baseline với mô hình; báo MAE/RMSE/MAPE và xử lý MAPE khi actual bằng 0. Gợi ý nhập hàng cần dùng forecast cùng tồn hiện tại, lead time, safety stock; các trường tồn/lead-time ở bộ mapping hiện chỉ là thông số minh họa.